In [1]:
import pandas as pd, numpy as np, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix)

import sys
sys.path.insert(0, '../')
from util import *     # preprocess_scenario, Timer, count_parameters, etc.
from models.transformer_lstm import Transformer_LSTM

device = "cuda:1" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(device) if "cuda" in device else device)

SCENARIOS = [f"../CSV_Files/scene{i+1}/" for i in range(6)]
MODEL_CLS  = Transformer_LSTM
MODEL_NAME = MODEL_CLS.__name__


device: NVIDIA A30


In [2]:
def to_tensors(arr):
    X = torch.from_numpy(arr[:, 1:14].astype("float32")).unsqueeze(1)
    y = torch.from_numpy(arr[:,  -1].astype("int64"))
    return X, y


def load_scenario(scenario_dir):
    train = pd.read_csv(scenario_dir + "train.csv").to_numpy()
    val   = pd.read_csv(scenario_dir + "val.csv").to_numpy()
    test  = pd.read_csv(scenario_dir + "test.csv").to_numpy()
    X_tr_n, X_va_n, X_te_n, _, _ = preprocess_scenario(train, val, test)
    return to_tensors(X_tr_n), to_tensors(X_va_n), to_tensors(X_te_n)


def make_model(model_cls, device, seed=0, **kwargs):
    """Fresh Transformer-LSTM with Xavier init on Linear layers.
    LSTM and MultiheadAttention use their own PyTorch defaults."""
    torch.manual_seed(seed)
    model = model_cls(n_classes=8, **kwargs).to(device)
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None: nn.init.zeros_(m.bias)
    return model


def make_loaders(X_tr, y_tr, X_va, y_va, X_te, y_te, batch_size=50, seed=0):
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(X_tr), generator=g)
    X_tr_s, y_tr_s = X_tr[perm], y_tr[perm]
    train_loader = DataLoader(TensorDataset(X_tr_s, y_tr_s),
                              batch_size=batch_size, shuffle=False)
    val_loader   = DataLoader(TensorDataset(X_va, y_va), batch_size=batch_size)
    test_loader  = DataLoader(TensorDataset(X_te, y_te), batch_size=batch_size)
    return train_loader, val_loader, test_loader


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = total_correct = total_n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss    += loss.item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_n       += xb.size(0)
    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def evaluate_loader(model, loader, criterion, device):
    model.eval()
    total_loss = total_correct = total_n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        total_loss    += criterion(logits, yb).item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_n       += xb.size(0)
    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def predict_all(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    for xb, yb in loader:
        logits = model(xb.to(device))
        y_pred.append(logits.argmax(1).cpu().numpy())
        y_true.append(yb.numpy())
    return np.concatenate(y_true), np.concatenate(y_pred)


def run_scenario(scenario_idx, scenario_dir, model_cls, device,
                 epochs=70, lr=1e-2, weight_decay=1e-4, seed=0, verbose=True):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = load_scenario(scenario_dir)
    train_loader, val_loader, test_loader = make_loaders(
        X_tr, y_tr, X_va, y_va, X_te, y_te, batch_size=50, seed=seed)

    model = make_model(model_cls, device, seed=seed)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr,
                           betas=(0.9, 0.999), eps=1e-8,
                           weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

    # --- compute metrics setup ---
    n_params, params_by_type = count_parameters(model)
    reset_gpu_peak_memory(device)

    if verbose:
        print(f"\n=== Scenario {scenario_idx} ({model_cls.__name__}) ===")
        print(f"  shapes: train={tuple(X_tr.shape)} val={tuple(X_va.shape)} "
              f"test={tuple(X_te.shape)}")
        print(f"  train labels: {np.bincount(y_tr.numpy())}")
        print(f"  params: {n_params:,}  breakdown: {params_by_type}")

    # --- TRAINING with timer ---
    with Timer(device) as train_timer:
        for ep in range(1, epochs + 1):
            tr_loss, tr_acc = train_one_epoch(model, train_loader,
                                              criterion, optimizer, device)
            va_loss, va_acc = evaluate_loader(model, val_loader,
                                              criterion, device)
            scheduler.step()
            if verbose and (ep == 1 or ep % 10 == 0 or ep == epochs):
                print(f"  ep {ep:3d} | train loss {tr_loss:.4f} acc {tr_acc:.3f}"
                      f" | val loss {va_loss:.4f} acc {va_acc:.3f}")

    peak_mem_mb = get_gpu_peak_memory_mb(device)

    # --- INFERENCE timing ---
    X_te_dev = X_te.to(device)
    def _predict(X):
        model.eval()
        with torch.no_grad():
            return model(X).argmax(1)
    inf_stats = measure_inference_time(_predict, X_te_dev, device,
                                       n_warmup=5, n_runs=20)

    # --- TEST accuracy ---
    y_true, y_pred = predict_all(model, test_loader, device)
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(8)))

    if verbose:
        print(f"  TEST: acc={acc:.4f}  P={p:.4f}  R={r:.4f}  F1={f:.4f}")
        print(f"  COMPUTE: train={train_timer.elapsed:.1f}s  "
              f"inf={inf_stats['per_sample_ms']:.3f}ms/sample  "
              f"peak_mem={peak_mem_mb:.1f}MB")

    return {
        "scenario":    scenario_idx,
        "model":       model_cls.__name__,
        "n_train":     len(X_tr),
        "accuracy":    acc,
        "precision":   p,
        "recall":      r,
        "f1":          f,
        "confusion":   cm,
        "n_params":    n_params,
        "train_sec":   round(train_timer.elapsed, 2),
        "inf_ms_per_sample": round(inf_stats["per_sample_ms"], 4),
        "peak_mem_mb": round(peak_mem_mb, 1),
    }


In [3]:
results = []
for i, sc_dir in enumerate(SCENARIOS, start=1):
    r = run_scenario(i, sc_dir, MODEL_CLS, device,
                     epochs=70, lr=1e-3, seed=0)
    results.append(r)


/home/maddie/miniconda3/envs/ad/lib/python3.12/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")



=== Scenario 1 (Transformer_LSTM) ===
  shapes: train=(7858, 1, 13) val=(1600, 1, 13) test=(1600, 1, 13)
  train labels: [ 858 1000 1000 1000 1000 1000 1000 1000]
  params: 5,339  breakdown: {'pos_emb': 39, 'input_proj': 6, 'transformer': 294, 'lstm': 4736, 'fc': 264}
  ep   1 | train loss 2.0651 acc 0.147 | val loss 1.9767 acc 0.177
  ep  10 | train loss 1.7431 acc 0.313 | val loss 1.5496 acc 0.371
  ep  20 | train loss 1.2139 acc 0.502 | val loss 1.1798 acc 0.519
  ep  30 | train loss 0.4276 acc 0.816 | val loss 0.4148 acc 0.849
  ep  40 | train loss 0.3391 acc 0.848 | val loss 0.3538 acc 0.858
  ep  50 | train loss 0.3204 acc 0.859 | val loss 0.3403 acc 0.866
  ep  60 | train loss 0.3065 acc 0.865 | val loss 0.3284 acc 0.871
  ep  70 | train loss 0.2975 acc 0.866 | val loss 0.3229 acc 0.872
  TEST: acc=0.8669  P=0.8671  R=0.8669  F1=0.8668
  COMPUTE: train=132.7s  inf=0.002ms/sample  peak_mem=27.3MB


/home/maddie/miniconda3/envs/ad/lib/python3.12/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")



=== Scenario 2 (Transformer_LSTM) ===
  shapes: train=(3939, 1, 13) val=(1600, 1, 13) test=(1600, 1, 13)
  train labels: [439 500 500 500 500 500 500 500]
  params: 5,339  breakdown: {'pos_emb': 39, 'input_proj': 6, 'transformer': 294, 'lstm': 4736, 'fc': 264}
  ep   1 | train loss 2.0799 acc 0.139 | val loss 2.0710 acc 0.172
  ep  10 | train loss 1.9182 acc 0.207 | val loss 1.9212 acc 0.235
  ep  20 | train loss 1.4978 acc 0.403 | val loss 1.4195 acc 0.425
  ep  30 | train loss 1.2940 acc 0.477 | val loss 1.3199 acc 0.474
  ep  40 | train loss 1.2523 acc 0.493 | val loss 1.2722 acc 0.496
  ep  50 | train loss 1.2204 acc 0.497 | val loss 1.2429 acc 0.495
  ep  60 | train loss 1.2135 acc 0.501 | val loss 1.2356 acc 0.504
  ep  70 | train loss 1.2038 acc 0.519 | val loss 1.2280 acc 0.504
  TEST: acc=0.5169  P=0.6076  R=0.5169  F1=0.5181
  COMPUTE: train=68.1s  inf=0.001ms/sample  peak_mem=27.3MB


/home/maddie/miniconda3/envs/ad/lib/python3.12/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


...

  ep   1 | train loss 2.0882 acc 0.097 | val loss 2.0811 acc 0.126
  ep  10 | train loss 2.0745 acc 0.166 | val loss 2.0719 acc 0.172
  ep  20 | train loss 2.0266 acc 0.156 | val loss 1.9814 acc 0.182
  ep  30 | train loss 1.9752 acc 0.161 | val loss 1.9650 acc 0.181
  ep  40 | train loss 1.9701 acc 0.179 | val loss 1.9613 acc 0.186
  ep  50 | train loss 1.9696 acc 0.174 | val loss 1.9603 acc 0.190
  ep  60 | train loss 1.9687 acc 0.205 | val loss 1.9598 acc 0.190
  ep  70 | train loss 1.9658 acc 0.205 | val loss 1.9597 acc 0.197
  TEST: acc=0.2019  P=0.3292  R=0.2019  F1=0.1426
  COMPUTE: train=13.3s  inf=0.002ms/sample  peak_mem=27.3MB


/home/maddie/miniconda3/envs/ad/lib/python3.12/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")



=== Scenario 5 (Transformer_LSTM) ===
  shapes: train=(238, 1, 13) val=(1600, 1, 13) test=(1600, 1, 13)
  train labels: [28 30 30 30 30 30 30 30]
  params: 5,339  breakdown: {'pos_emb': 39, 'input_proj': 6, 'transformer': 294, 'lstm': 4736, 'fc': 264}
  ep   1 | train loss 2.0866 acc 0.118 | val loss 2.0820 acc 0.126
  ep  10 | train loss 2.0833 acc 0.147 | val loss 2.0770 acc 0.148
  ep  20 | train loss 2.0700 acc 0.151 | val loss 2.0699 acc 0.173
  ep  30 | train loss 2.0711 acc 0.160 | val loss 2.0620 acc 0.177
  ep  40 | train loss 2.0610 acc 0.151 | val loss 2.0420 acc 0.179
  ep  50 | train loss 2.0395 acc 0.143 | val loss 2.0188 acc 0.179
  ep  60 | train loss 2.0147 acc 0.185 | val loss 2.0005 acc 0.177
  ep  70 | train loss 2.0279 acc 0.189 | val loss 1.9905 acc 0.176
  TEST: acc=0.1875  P=0.0877  R=0.1875  F1=0.0972
  COMPUTE: train=11.1s  inf=0.001ms/sample  peak_mem=27.3MB

=== Scenario 6 (Transformer_LSTM) ===
  shapes: train=(77, 1, 13) val=(1600, 1, 13) test=(1600, 1, 1

/home/maddie/miniconda3/envs/ad/lib/python3.12/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


  ep   1 | train loss 2.0870 acc 0.104 | val loss 2.0860 acc 0.124
  ep  10 | train loss 2.0815 acc 0.117 | val loss 2.0792 acc 0.134
  ep  20 | train loss 2.0763 acc 0.182 | val loss 2.0802 acc 0.135
  ep  30 | train loss 2.0731 acc 0.143 | val loss 2.0813 acc 0.128
  ep  40 | train loss 2.0560 acc 0.182 | val loss 2.0832 acc 0.126
  ep  50 | train loss 2.0662 acc 0.182 | val loss 2.0832 acc 0.130
  ep  60 | train loss 2.0728 acc 0.143 | val loss 2.0833 acc 0.128
  ep  70 | train loss 2.0687 acc 0.169 | val loss 2.0832 acc 0.128
  TEST: acc=0.1250  P=0.1406  R=0.1250  F1=0.0290
  COMPUTE: train=8.7s  inf=0.001ms/sample  peak_mem=27.3MB


In [4]:
summary = pd.DataFrame([
    {k: r[k] for k in ("scenario", "model", "n_train",
                       "accuracy", "precision", "recall", "f1",
                       "n_params", "train_sec", "inf_ms_per_sample",
                       "peak_mem_mb")}
    for r in results
])
cols_round = ["accuracy", "precision", "recall", "f1"]
summary[cols_round] = summary[cols_round].round(4)
print(f"\n=== {MODEL_NAME} summary across scenarios ===")
print(summary.to_string(index=False))

summary.to_csv(f"{MODEL_NAME.lower()}_results_by_scenario.csv", index=False)



=== Transformer_LSTM summary across scenarios ===
 scenario            model  n_train  accuracy  precision  recall     f1  n_params  train_sec  inf_ms_per_sample  peak_mem_mb
        1 Transformer_LSTM     7858    0.8669     0.8671  0.8669 0.8668      5339     132.65             0.0015         27.3
        2 Transformer_LSTM     3939    0.5169     0.6076  0.5169 0.5181      5339      68.15             0.0014         27.3
        3 Transformer_LSTM      786    0.2544     0.3589  0.2544 0.2294      5339      19.84             0.0015         27.3
        4 Transformer_LSTM      391    0.2019     0.3292  0.2019 0.1426      5339      13.26             0.0023         27.3
        5 Transformer_LSTM      238    0.1875     0.0877  0.1875 0.0972      5339      11.15             0.0015         27.3
        6 Transformer_LSTM       77    0.1250     0.1406  0.1250 0.0290      5339       8.74             0.0015         27.3


In [5]:
for r in results:
    print(f"\nScenario {r['scenario']}  ({r['model']}, "
          f"n_train={r['n_train']}, acc={r['accuracy']:.3f})")
    print(r["confusion"])



Scenario 1  (Transformer_LSTM, n_train=7858, acc=0.867)
[[149   0   0  50   0   0   1   0]
 [  1 182   0   1   0   1   0  15]
 [  0   0 196   0   0   4   0   0]
 [ 39   0   0 159   0   0   2   0]
 [  0   0   0   0 200   0   0   0]
 [  0   2   4   0   0 156   0  38]
 [  0   0   0   1   1   0 198   0]
 [  0  13   0   0   0  40   0 147]]

Scenario 2  (Transformer_LSTM, n_train=3939, acc=0.517)
[[ 15   2  35  89   3   0   7  49]
 [ 10  78  49  15   0   0   0  48]
 [  0   1 144   0   0   0   0  55]
 [ 17   7  32  96   1   0  14  33]
 [  0   0   0   0 200   0   0   0]
 [  0   2  49   3   0  97   0  49]
 [ 13   5  13  49   1   0  99  20]
 [  2   3  94   3   0   0   0  98]]

Scenario 3  (Transformer_LSTM, n_train=786, acc=0.254)
[[  0   0   4   0   1   1  69 125]
 [  0  74   1   0   0   8  29  88]
 [  0   0   3   0   7   1  47 142]
 [  0   1   2   0   0   3  70 124]
 [  0   0   2   0  36   1  97  64]
 [  0  16   0   0   3  79  33  69]
 [  0   0   6   0   1   0  64 129]
 [  0   0   1   0   0  